In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

print(df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())  # important - check for missing values

(891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN    

In [2]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.Cabin.nunique()

147

In [4]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [5]:
df = df.dropna(subset=['Embarked'])

In [6]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64

In [7]:
df['Age'] = df['Age'].fillna(df['Age'].median())

In [8]:
df = df.drop(columns=['Cabin', 'PassengerId', 'Name', 'Ticket'])

In [9]:
df.info()

<class 'pandas.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  889 non-null    int64  
 1   Pclass    889 non-null    int64  
 2   Sex       889 non-null    str    
 3   Age       889 non-null    float64
 4   SibSp     889 non-null    int64  
 5   Parch     889 non-null    int64  
 6   Fare      889 non-null    float64
 7   Embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 62.5 KB


In [10]:
print(df['SibSp'].value_counts())

SibSp
0    606
1    209
2     28
4     18
3     16
8      7
5      5
Name: count, dtype: int64


In [11]:
df = pd.get_dummies(df, columns=['Embarked'], dtype=int)

In [ ]:
df.head()

In [12]:
df.dtypes

Survived        int64
Pclass          int64
Sex               str
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked_C      int64
Embarked_Q      int64
Embarked_S      int64
dtype: object

In [ ]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})  # binary encode

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

In [ ]:
x = df.drop('Survived', axis=1).values  # all columns except Survived, as a numpy array
y = df['Survived'].values                # just the Survived column, as a numpy array

In [ ]:
from sklearn.utils import shuffle

x_train, y_train = shuffle(x_train, y_train, random_state=42)  # shuffle before k-fold

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42  # 80% train, 20% test, fixed seed for reproducibility
)

In [ ]:
x_train.shape

In [ ]:
print(df.dtypes)

In [ ]:
x_train.dtype

In [ ]:
print(pd.DataFrame(x_train).dtypes)

In [ ]:
mean = x_train.mean(axis=0)
std = x_train.std(axis=0)
x_train = (x_train - mean) / std
x_test = (x_test- mean) / std

In [ ]:
import keras
from keras import layers

def get_model():
    model = keras.Sequential(
        [
            layers.Dense(18, activation="sigmoid"),
            layers.Dense(18, activation="sigmoid"),
            layers.Dense(1),
        ]
    )
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
from keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',    # watch the validation loss
    patience=10,           # stop if no improvement for 10 epochs
    restore_best_weights=True  # revert to the best weights when stopping
)

In [ ]:
import numpy as np

k = 5
num_val_samples = len(x_train) // k
num_epochs = 100
all_accu_histories = []
for i in range(k):
    print(f"Processing fold #{i + 1}")
    fold_x_val = x_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_y_val = y_train[i * num_val_samples : (i + 1) * num_val_samples]
    fold_x_train = np.concatenate(
        [x_train[: i * num_val_samples], x_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    fold_y_train = np.concatenate(
        [y_train[: i * num_val_samples], y_train[(i + 1) * num_val_samples :]],
        axis=0,
    )
    model = get_model()
    history = model.fit(
        fold_x_train,
        fold_y_train,
        epochs=num_epochs,
        batch_size=16,
        verbose=0,
        validation_data=(fold_x_val, fold_y_val),
        callbacks=[early_stopping]
    )
    accu_history = history.history["accuracy"]
    all_accu_histories.append(accu_history)


In [ ]:
min_epochs = min(len(x) for x in all_accu_histories)  # find shortest fold

average_accu_history = [
    np.mean([x[i] for x in all_accu_histories]) for i in range(min_epochs)  # only go up to min length
]

In [ ]:
#%matplotlib widget
import matplotlib.pyplot as plt
plt.clf()

epochs = range(1, len(average_accu_history) + 1)
plt.plot(epochs, average_accu_history)
plt.xlabel("Epochs")
plt.ylabel("Validation Accu")
plt.show()

In [ ]:
print(np.isnan(x_train).sum())  # any NaNs left?
print(np.isnan(y_train).sum())

In [ ]:
all_accu_histories